# 1 - Variational auto-encoder


> Part of **[Compressed Sensing using Generative Models](../README.md)**.
> Run `pip install -e .` from the repository root first; every notebook imports
> the `csgm` package rather than redefining the model and recovery code inline.


A VAE learns a smooth, low-dimensional description of the MNIST manifold. Two
networks are trained jointly:

- the **encoder** $q_\phi(z \mid x)$ maps an image to the parameters
  $(\mu, \log \sigma^2)$ of a diagonal Gaussian in $\mathbb{R}^k$;
- the **decoder** $p_\theta(x \mid z)$ maps a latent code back to an image.

Training maximises the evidence lower bound

$$\mathcal{L}(\theta, \phi; x) = \underbrace{\mathbb{E}_{q_\phi(z \mid x)}\big[\log p_\theta(x \mid z)\big]}_{\text{reconstruction}} - \underbrace{D_{\mathrm{KL}}\big(q_\phi(z \mid x) \,\|\, p(z)\big)}_{\text{regularisation}},$$

with the prior $p(z) = \mathcal{N}(0, I)$. The KL term is what makes the VAE
useful here: it forces the aggregate posterior towards the standard normal, so
at recovery time we can safely search over $z \sim \mathcal{N}(0, I)$ and expect
the decoder to answer with a plausible digit.

**Only the decoder is needed downstream** - it is the generator $G$ of
[notebook 4](04_compressed_sensing_recovery.ipynb).

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np

from csgm.config import DEFAULT_SEED, MODELS_DIR
from csgm.data import load_mnist
from csgm.models import VAE, build_decoder, build_encoder
from csgm.viz import show_images

keras.utils.set_random_seed(DEFAULT_SEED)
print("keras", keras.__version__)

## 1.1 Data

Pixels are scaled to $[0, 1]$ to match the decoder's sigmoid output.

In [ ]:
(x_train, y_train), (x_test, y_test) = load_mnist()
print("train", x_train.shape, "| test", x_test.shape, "| range", (x_train.min(), x_train.max()))

show_images(x_train[:8], [str(d) for d in y_train[:8]], suptitle="MNIST training samples");

## 1.2 Architecture

The encoder is four convolutions (32, 64, 64, 64 filters, one of them strided)
followed by a dense bottleneck; the decoder mirrors it with two transposed
convolutions. `latent_dim` is the only knob that matters for the compressed
sensing experiment - the shipped checkpoints use $k = 20$ and $k = 30$.

Set `latent_dim = 2` instead to reproduce the latent-space scatter plot at the
end of this notebook.

In [ ]:
latent_dim = 20

encoder = build_encoder(latent_dim)
decoder = build_decoder(latent_dim)
encoder.summary()
decoder.summary()

### The reparameterisation trick

Sampling $z \sim \mathcal{N}(\mu, \sigma^2)$ is not differentiable with respect
to $\mu$ and $\sigma$. Writing $z = \mu + \sigma \odot \epsilon$ with
$\epsilon \sim \mathcal{N}(0, I)$ moves the randomness to a leaf of the graph, so
gradients reach the encoder. That is the whole job of `csgm.models.Sampling`.

In [ ]:
from csgm.models import Sampling

probe = Sampling()([np.zeros((5, latent_dim), "float32"), np.zeros((5, latent_dim), "float32")])
print("five draws from N(0, I):", np.asarray(probe)[:, 0])

## 1.3 Training

100 epochs at batch size 100 with Adam ($10^{-3}$) is what produced the shipped
checkpoints; the cell below uses far fewer so the notebook stays runnable. For
the real thing use the script, which also handles early stopping:

```bash
python scripts/train_vae.py --latent-dim 20 --epochs 100
```

In [ ]:
EPOCHS = 5  # -> 100 to reproduce the checkpoints

vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3))
history = vae.fit(
    x_train,
    epochs=EPOCHS,
    batch_size=100,
    validation_data=(x_test,),
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)],
)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
for key in ("loss", "reconstruction_loss", "kl_loss"):
    ax.plot(history.history[key], marker="o", ms=3, label=key.replace("_", " "))
ax.set_xlabel("epoch")
ax.set_ylabel("negative ELBO (nats)")
ax.set_title("VAE training")
ax.grid(alpha=0.3)
ax.legend(frameon=False);

The reconstruction term dominates the total loss, while the KL term settles on a
small positive plateau: the posterior stays close enough to the prior for latent
search to work, without collapsing onto it (which would make the decoder ignore
$z$ entirely).

## 1.4 What the decoder learned

In [ ]:
reconstructed = vae.predict(x_test[:8], verbose=0)
show_images(
    np.concatenate([x_test[:8], reconstructed]),
    ["original"] * 8 + ["reconstruction"] * 8,
    suptitle="Auto-encoding held-out digits",
);

In [ ]:
z = np.random.default_rng(0).standard_normal((16, latent_dim)).astype("float32")
show_images(decoder(z, training=False), suptitle="Unconditional samples: decoding z ~ N(0, I)");

These samples are the *entire hypothesis space* of the compressed sensing
recovery in notebook 4: it can only ever return an image the decoder is able to
produce. Blurry samples here mean a blurry error floor there.

## 1.5 Structure of the latent space

With $k = 2$ the posterior means can be plotted directly. Digits form
well-separated clusters, which is why moving through the latent space produces
smooth morphs between classes rather than noise - exactly the structure that
gradient descent on $z$ exploits.

<img src="../docs/figures/vae_latent_space_2d.png" width="620">

Re-run this notebook with `latent_dim = 2` to regenerate the figure:

```python
mu, _, _ = encoder.predict(x_test)
plt.scatter(mu[:, 0], mu[:, 1], c=y_test, cmap="brg", s=2)
```

## 1.6 Save the decoder

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
path = MODELS_DIR / f"vae_decoder_dim{latent_dim}.keras"
# decoder.save(path)   # uncomment to overwrite the shipped checkpoint
print("would save to", path)

---
Next: [2 - DCGAN](02_dcgan_training.ipynb), the adversarially trained
alternative prior.